# Real-World Conflation: Sundbyberg Road Map Matching

Welcome to the Sundbyberg map conflation matching dashboard! This notebook is configured to match and conflate your Sundbyberg road networks using local, self-contained datasets:
1. **OSM Traffic Enrichment Edges**: `data/osm_edges.csv`
2. **Sweden NVDB Standardized Directed Edges**: `data/sweden_edges.csv`

### 💡 GitHub-Ready & Portable Data
Both road networks have been pre-processed and exported as standard CSV files directly inside the project's `data/` directory. All coordinates and spatial geometries are stored as Well-Known Text (WKT). 

When you run this notebook, DuckDB Spatial will automatically parse the local WKT files into active in-memory geometry tables. This makes the entire repository fully portable, ready for GitHub, and immune to any read-write database file locking conflicts!

## Step 1: Inspect Database Schemas (Read-Only)

You can run this cell to verify all tables and columns present in both databases. We use `read_only=True` connections to prevent any file-locking conflicts with other active sessions.

In [ ]:
import pandas as pd

print("==================================================")
print("         INSPECTING LOCAL DATASETS (CSV)")
print("==================================================")

print("--> OSM Road Network (osm_edges.csv):")
df_osm = pd.read_csv("../data/osm_edges.csv")
display(df_osm.head())

print("\n--> Sweden Road Network (sweden_edges.csv):")
df_sweden = pd.read_csv("../data/sweden_edges.csv")
display(df_sweden.head())

## Step 2: Configured Table & Column Names

The tables and columns are configured as follows:
* **Source A**: OSM directed edges table is `driving.edges`.
* **Source B**: Preprocessed directed Sweden edges table is `main.vehicle_edges_directed` (generated inside Database A during Step 3).
* **Projection System**: Projected into **SWEREF99 TM (EPSG:3006)**—Sweden's national metric coordinate system.

In [ ]:
# Config from DB A (OSM)
TABLE_A = "driving_edges"
ID_A = "edge_id"
GEOM_A = "geometry"

# Config from DB B (Sweden NVDB preprocessed table)
TABLE_B = "vehicle_edges_directed"
ID_B = "directed_id"
GEOM_B = "geometry"

# Local projected coordinate system in meters for Sweden (SWEREF99 TM)
UTM_SRID = 3006

print("✅ Local table and column configurations prepared!")

## Step 3: Directed Preprocessing & Matcher Initialization

We initialize our `DuckDBMapMatcher`, dynamically construct the directed Sweden edges representation `main.vehicle_edges_directed`, configure the sources, and perform candidate generation.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from network_matching import DuckDBMapMatcher

# 1. Initialize matcher with a clean in-memory DuckDB connection
print("[Step 1] Initializing in-memory map matcher...")
matcher = DuckDBMapMatcher()

# 2. Load local WKT datasets and parse them into in-memory spatial tables
print("\n[Step 2] Loading local WKT datasets and parsing geometries...")
matcher.conn.execute("""
    CREATE OR REPLACE TABLE driving_edges AS
    SELECT 
        edge_id::BIGINT AS edge_id,
        ST_GeomFromText(geometry) AS geometry
    FROM '../data/osm_edges.csv';
""")

matcher.conn.execute("""
    CREATE OR REPLACE TABLE vehicle_edges_directed AS
    SELECT 
        directed_id::BIGINT AS directed_id,
        original_edge_id::BIGINT AS original_edge_id,
        name,
        is_reverse::BOOLEAN AS is_reverse,
        ST_GeomFromText(geometry) AS geometry
    FROM '../data/sweden_edges.csv';
""")

# 3. Configure column mapping
matcher.configure_sources(
    source_a=TABLE_A, id_col_a=ID_A, geom_col_a=GEOM_A,
    source_b=TABLE_B, id_col_b=ID_B, geom_col_b=GEOM_B,
    utm_srid=UTM_SRID
)

# 4. Set matching thresholds
#matcher.set_parameters(max_distance=25.0, max_angle=30.0, min_overlap=0.50)
matcher.set_parameters(max_distance=25.0)

# 5. Tier 1: Candidate Generation
print("\n[Step 3] Finding candidate overlaps using DuckDB Spatial index...")
candidates = matcher.generate_candidate_pairs()
print(f"--> Found {len(candidates)} candidate pairs to evaluate.")

## Step 4: Run DTW Alignments & Reconcile

We evaluate shape similarity on the candidate list using Continuous Subsequence DTW and reconcile matching decisions in SQL.

In [ ]:
if not candidates.empty:
    # 5. Tier 2: DTW Shape matching
    print("[Step 4] Running 2D Subsequence DTW shape matching...")
    evaluated = matcher.compute_dtw_metrics(candidates)
    
    # 6. Tier 3: SQL Reconciliation & Split Road Detection
    print("\n[Step 5] Reconciling matches in SQL (Symmetric, Splits, Conflicts)...")
    results = matcher.reconcile_matches(evaluated)
    
    print("\nConflation Matches Summary:")
    print(results["match_type"].value_counts())
    
    # Display top 10 matches
    print("\nSample Matches Table:")
    display(results.head(10))
else:
    print("No candidates to match.")


## Step 5: Interactive Map Visualization

Let's see our matches visualized on an interactive dark-matter Leaflet map! We load the original geometries, overlay the matching decisions, and highlight match categories: 
* **Green**: Precise symmetric 1:1 matches.
* **Blue**: Coarse-to-fine 1:N splits.
* **Orange**: Directional conflicts or unidirectionally partial alignments.
* **Dashed Gray**: Unmatched OSM roads.
* **Dashed Red**: Unmatched Sweden NVDB roads.

In [ ]:
def visualize_matching_results(matcher, results_df, offset_deg=0.00018, strategy="best_per_source"):
    """
    Draw BOTH road networks on one interactive map for side-by-side comparison.

    - Network A (OSM) is drawn in its true position.
    - Network B (Sweden) is drawn SHIFTED by `offset_deg` (in degrees, north-east) so the
      two networks sit beside each other instead of perfectly overlapping.
    - Matches are decided with `resolve(strategy=...)` (default best_per_source = each OSM
      road -> its single closest Sweden road), so the DTW shown is the BEST per road, not a
      pile of weak candidates.
    - Thin orange "match links" connect each matched A road to its (shifted) B partner.

    Tweak `offset_deg` to separate the networks more/less (0.00018 deg ~ 10-20 m here).
    """
    import folium
    import geopandas as gpd
    import pandas as pd
    from shapely.wkt import loads as load_wkt
    from shapely.affinity import translate

    print("Fetching road geometries...")
    df_a = matcher.conn.execute(
        f"SELECT {matcher.columns_a['id']} AS id_a, ST_AsText({matcher.columns_a['geom']}) AS wkt FROM {matcher.source_a}"
    ).df()
    df_b = matcher.conn.execute(
        f"SELECT {matcher.columns_b['id']} AS id_b, name, is_reverse, ST_AsText({matcher.columns_b['geom']}) AS wkt FROM {matcher.source_b}"
    ).df()
    gdf_a = gpd.GeoDataFrame(df_a, geometry=df_a["wkt"].apply(load_wkt), crs="EPSG:4326").drop(columns=["wkt"])
    gdf_b = gpd.GeoDataFrame(df_b, geometry=df_b["wkt"].apply(load_wkt), crs="EPSG:4326").drop(columns=["wkt"])

    # Decide one best match per OSM road (many-to-one) so each road shows its closest partner.
    assignment = matcher.resolve(results_df, strategy=strategy)
    matched = assignment[assignment["match_type"] != "NO_MATCH"].copy()
    a_meta = matched.drop_duplicates("source_id").set_index("source_id")  # 1 row per source
    matched_src = set(matched["source_id"])
    matched_dst = set(matched["dest_id"])

    # Offset network B so it sits beside A instead of overlapping it.
    gdf_b["geom_shift"] = gdf_b["geometry"].apply(lambda g: translate(g, xoff=offset_deg, yoff=offset_deg))

    print("Building map...")
    bounds = gdf_a.total_bounds
    m = folium.Map(
        location=[(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2],
        zoom_start=15, tiles="CartoDB dark_matter",
    )

    fg_a = folium.FeatureGroup(name="🟩 OSM network A (green=matched, gray=unmatched)", show=True)
    fg_b = folium.FeatureGroup(name="🟦 Sweden network B, offset (blue=matched, red=unmatched)", show=True)
    fg_links = folium.FeatureGroup(name="⚡ Match links (A → B)", show=False)

    def tip(html):
        # sticky + nowrap so the FULL row of fields always shows and never clips/wraps.
        return folium.Tooltip(
            f"<div style='font-family: Arial, sans-serif; font-size: 12px; white-space: nowrap;'>{html}</div>",
            sticky=True,
        )

    # --- Network A (OSM), true position ---
    for _, r in gdf_a.iterrows():
        sid = r["id_a"]
        if sid in matched_src:
            mt = a_meta.loc[sid]
            color, dash = "#10b981", None  # green
            html = (f"<b>OSM A:</b> {sid}<br>"
                    f"<b>→ Sweden B:</b> {mt['dest_id']}<br>"
                    f"<b>Match type:</b> {mt['match_type']}<br>"
                    f"<b>DTW distance:</b> {mt['dtw_distance']:.1f} m<br>"
                    f"<b>Bearing Δ:</b> {mt['bearing_diff']:.0f}°<br>"
                    f"<b>Overlap:</b> {int(mt['overlap_pct'])}%")
        else:
            color, dash = "#64748b", "4, 6"  # gray dashed
            html = f"<b>OSM A:</b> {sid}<br><b>Status:</b> NO_MATCH"
        folium.GeoJson(
            r["geometry"].__geo_interface__,
            style_function=lambda x, c=color, d=dash: {"color": c, "weight": 3, "opacity": 0.9, "dashArray": d},
            tooltip=tip(html),
        ).add_to(fg_a)

    # --- Network B (Sweden), offset ---
    for _, r in gdf_b.iterrows():
        did = r["id_b"]
        used = did in matched_dst
        color, dash = ("#3b82f6", None) if used else ("#ef4444", "4, 6")  # blue / red dashed
        dir_label = "Reverse" if r["is_reverse"] else "Forward"
        html = (f"<b>Sweden B:</b> {did}<br>"
                f"<b>Name:</b> {r['name']}<br>"
                f"<b>Direction:</b> {dir_label}<br>"
                f"<b>Status:</b> {'matched' if used else 'NO_MATCH'}")
        folium.GeoJson(
            r["geom_shift"].__geo_interface__,
            style_function=lambda x, c=color, d=dash: {"color": c, "weight": 3, "opacity": 0.9, "dashArray": d},
            tooltip=tip(html),
        ).add_to(fg_b)

    # --- Match links: A midpoint → shifted-B midpoint ---
    a_idx = gdf_a.set_index("id_a")
    b_idx = gdf_b.set_index("id_b")
    for _, mt in matched.iterrows():
        try:
            pa = a_idx.loc[mt["source_id"], "geometry"].interpolate(0.5, normalized=True)
            pb = b_idx.loc[mt["dest_id"], "geom_shift"].interpolate(0.5, normalized=True)
        except (KeyError, AttributeError):
            continue
        folium.PolyLine([(pa.y, pa.x), (pb.y, pb.x)], color="#f59e0b", weight=1, opacity=0.5).add_to(fg_links)

    fg_a.add_to(m)
    fg_b.add_to(m)
    fg_links.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

    n_m = len(matched_src)
    print(f"✅ Map ready — {n_m} matched OSM roads, {len(gdf_a) - n_m} unmatched; "
          f"network B offset by {offset_deg}°.")
    return m

# Call the visualization function and show the map
if not results.empty:
    conflation_map = visualize_matching_results(matcher, results)

    # Save the interactive map to a standalone HTML file you can open in a browser
    html_out_path = "../output/conflation_map.html"
    conflation_map.save(html_out_path)
    print(f"✅ Saved interactive map to: {html_out_path}")

    display(conflation_map)
else:
    print("No results to visualize.")

## Step 6: Export Conflation Table

Write the final matching results back to a new table inside your primary Sundbyberg database, or save it to a flat GeoPackage/CSV file.

In [ ]:
if not candidates.empty:
    # Build the COMPLETE table: matched pairs + every OSM (Source A) segment that found
    # no match, appended as NO_MATCH rows (dest_id=None, metrics=NaN). This way the saved
    # file lists every OSM segment, matched or not -- not just the ones that matched.
    results_full = matcher._append_unmatched(results, matcher._get_all_ids_a())

    n_matched = (results_full["match_type"] != "NO_MATCH").sum()
    n_nomatch = (results_full["match_type"] == "NO_MATCH").sum()
    print(f"Rows: {n_matched} matched + {n_nomatch} NO_MATCH = {len(results_full)} total")

    # Register the combined DataFrame in DuckDB connection
    matcher.conn.register("final_matches", results_full)

    # Save the permanent table as a local CSV inside the project data directory
    csv_out_path = "../output/conflation_results.csv"
    matcher.conn.execute(f"COPY final_matches TO '{csv_out_path}' (HEADER, DELIMITER ',');")
    print(f"✅ Successfully saved conflation results to local CSV: {csv_out_path}")

    matcher.conn.close()
else:
    print("No results to export.")

## Step 7: Symmetric (Two-Way) Split-Aware Matching

The cells above are **directed** (OSM A → Sweden B). This step runs `match_symmetric()`, which matches **both** ways (A→B and B→A) and reconciles them using *both* overlap values, so it keeps clean 1:1 matches **and** preserves split roads (1:N) and merges (N:1) — while dropping incidental crossings. See [`docs/symmetric_matching.md`](../docs/symmetric_matching.md).

In the map, network B is offset so the two networks sit side by side, and each **match link is colored by its cardinality**:

* 🟢 **1:1** — same road in both networks
* 🔵 **1:N split** — one OSM road = several Sweden pieces
* 🟠 **N:1 merge** — several OSM roads = one Sweden piece
* 🟣 **N:M complex** — tangled many-to-many cluster

Hover any link for the two overlap values, DTW, bearing difference, relation, and cardinality.

In [ ]:
# === Symmetric (two-way) split-aware matching: run + visualize ===
# Self-contained: builds its own matcher so it works regardless of earlier cells.
import folium
import geopandas as gpd
from shapely.wkt import loads as load_wkt
from shapely.affinity import translate
from network_matching import DuckDBMapMatcher

# 1. Fresh matcher + data
sm = DuckDBMapMatcher()
sm.conn.execute("""
    CREATE TABLE driving_edges AS
    SELECT edge_id::BIGINT AS edge_id, ST_GeomFromText(geometry) AS geometry
    FROM '../data/osm_edges.csv';
""")
sm.conn.execute("""
    CREATE TABLE vehicle_edges_directed AS
    SELECT directed_id::BIGINT AS directed_id, name, is_reverse::BOOLEAN AS is_reverse,
           ST_GeomFromText(geometry) AS geometry
    FROM '../data/sweden_edges.csv';
""")
sm.configure_sources(
    source_a="driving_edges", id_col_a="edge_id", geom_col_a="geometry",
    source_b="vehicle_edges_directed", id_col_b="directed_id", geom_col_b="geometry",
    utm_srid=3006,
)
sm.set_parameters(max_distance=25.0)

# 2. Symmetric matching (A->B and B->A reconciled with both overlaps)
# max_dtw=25 matches the candidate search radius (max_distance); the 12 m default was
# too strict for this dataset, where OSM and NVDB are offset by ~20 m on some streets.
sym = sm.match_symmetric(max_dtw=25.0, max_angle=45.0, min_overlap_m=5.0, sym_overlap=70)
print(f"Symmetric edges: {len(sym)}")
print(sym["cardinality"].value_counts())


def visualize_symmetric(matcher, sym, offset_deg=0.00018,
                        boundary_path="../data/sundbyberg_boundary.geojson",
                        osm_csv="../data/osm_edges.csv",
                        sweden_csv="../data/sweden_edges.csv"):
    """Both networks (B offset); match links colored by cardinality (1:1 / split / merge / complex)."""
    df_a = matcher.conn.execute(
        f"SELECT {matcher.columns_a['id']} AS id_a, ST_AsText({matcher.columns_a['geom']}) AS wkt FROM {matcher.source_a}"
    ).df()
    df_b = matcher.conn.execute(
        f"SELECT {matcher.columns_b['id']} AS id_b, name, is_reverse, ST_AsText({matcher.columns_b['geom']}) AS wkt FROM {matcher.source_b}"
    ).df()
    gdf_a = gpd.GeoDataFrame(df_a, geometry=df_a["wkt"].apply(load_wkt), crs="EPSG:4326").drop(columns=["wkt"]).set_index("id_a")
    gdf_b = gpd.GeoDataFrame(df_b, geometry=df_b["wkt"].apply(load_wkt), crs="EPSG:4326").drop(columns=["wkt"]).set_index("id_b")
    gdf_b["geom_shift"] = gdf_b["geometry"].apply(lambda g: translate(g, xoff=offset_deg, yoff=offset_deg))

    matched_a, matched_b = set(sym["a_id"]), set(sym["b_id"])
    CARD_COLORS = {"1:1": "#10b981", "1:N_SPLIT": "#3b82f6", "N:1_MERGE": "#f59e0b", "N:M_COMPLEX": "#a855f7"}

    bnds = gdf_a.total_bounds
    m = folium.Map(location=[(bnds[1] + bnds[3]) / 2, (bnds[0] + bnds[2]) / 2],
                   zoom_start=15, tiles=None)
    # Base layers — pick one in the layer control (top-right)
    folium.TileLayer("CartoDB dark_matter", name="Dark (CartoDB)").add_to(m)
    folium.TileLayer("CartoDB positron", name="Light (CartoDB)").add_to(m)
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery", name="Satellite (Esri)").add_to(m)

    fg_a = folium.FeatureGroup(name="▫️ OSM network A (white=matched)", show=True)
    fg_b = folium.FeatureGroup(name="▫️ Sweden network B, offset (gray=matched)", show=True)
    link_fgs = {c: folium.FeatureGroup(name=f"🔗 {c} links", show=True) for c in CARD_COLORS}

    def tip(html):
        return folium.Tooltip(
            f"<div style='font-family:Arial,sans-serif;font-size:12px;white-space:nowrap;'>{html}</div>",
            sticky=True,
        )

    # Network A (true position)
    for aid, r in gdf_a.iterrows():
        on = aid in matched_a
        color, dash, w = ("#e5e7eb", None, 2.5) if on else ("#475569", "3, 6", 1.5)
        folium.GeoJson(
            r["geometry"].__geo_interface__,
            style_function=lambda x, c=color, d=dash, w=w: {"color": c, "weight": w, "opacity": 0.8, "dashArray": d},
            tooltip=tip(f"<b>OSM A:</b> {aid}<br><b>In match:</b> {'yes' if on else 'no'}"),
        ).add_to(fg_a)

    # Network B (offset)
    for bid, r in gdf_b.iterrows():
        on = bid in matched_b
        color, dash, w = ("#94a3b8", None, 2.5) if on else ("#7f1d1d", "3, 6", 1.5)
        dlab = "Reverse" if r["is_reverse"] else "Forward"
        folium.GeoJson(
            r["geom_shift"].__geo_interface__,
            style_function=lambda x, c=color, d=dash, w=w: {"color": c, "weight": w, "opacity": 0.8, "dashArray": d},
            tooltip=tip(f"<b>Sweden B:</b> {bid}<br><b>Name:</b> {r['name']}<br><b>Dir:</b> {dlab}<br><b>In match:</b> {'yes' if on else 'no'}"),
        ).add_to(fg_b)

    # Match links, colored by cardinality, grouped into toggleable layers
    for _, e in sym.iterrows():
        try:
            pa = gdf_a.loc[e["a_id"], "geometry"].interpolate(0.5, normalized=True)
            pb = gdf_b.loc[e["b_id"], "geom_shift"].interpolate(0.5, normalized=True)
        except (KeyError, AttributeError):
            continue
        card = e["cardinality"]
        html = (f"<b>{e['a_id']} &harr; {e['b_id']}</b><br>"
                f"<b>Relation:</b> {e['relation']}<br>"
                f"<b>Cardinality:</b> {card}<br>"
                f"<b>overlap a&rarr;b:</b> {int(e['ov_ab'])}%  &nbsp; <b>b&rarr;a:</b> {int(e['ov_ba'])}%<br>"
                f"<b>DTW:</b> {e['dtw']:.1f} m  &nbsp; <b>Bearing &Delta;:</b> {e['bearing_diff']:.0f}&deg;")
        folium.PolyLine([(pa.y, pa.x), (pb.y, pb.x)],
                        color=CARD_COLORS.get(card, "#ffffff"), weight=2.5, opacity=0.85,
                        tooltip=tip(html)).add_to(link_fgs[card])

    fg_a.add_to(m)
    fg_b.add_to(m)
    for fg in link_fgs.values():
        fg.add_to(m)
    # Undirected (two-way) edges as opt-in filter layers (hidden by default)
    fg_osm_u = folium.FeatureGroup(name="\U0001f7e9 Undirected OSM edges (two-way)", show=False)
    fg_swe_u = folium.FeatureGroup(name="\U0001f7ea Undirected Sweden edges (two-way)", show=False)
    try:
        osm_u = matcher.conn.execute(
            f"SELECT geometry FROM '{osm_csv}' WHERE lower(CAST(is_reverse AS VARCHAR))='false' "
            f"AND NOT (lower(CAST(oneway AS VARCHAR)) IN ('yes','true','1','-1'))"
        ).fetchall()
        for (wkt,) in osm_u:
            folium.GeoJson(load_wkt(wkt).__geo_interface__,
                style_function=lambda x: {"color": "#2dd4bf", "weight": 1.5, "opacity": 0.7}).add_to(fg_osm_u)
    except Exception as _e:
        print("undirected OSM layer skipped:", _e)
    try:
        swe_u = matcher.conn.execute(
            f"WITH cnt AS (SELECT original_edge_id FROM '{sweden_csv}' GROUP BY original_edge_id HAVING COUNT(*)=2) "
            f"SELECT s.geometry FROM '{sweden_csv}' s JOIN cnt USING(original_edge_id) "
            f"WHERE lower(CAST(s.is_reverse AS VARCHAR))='false'"
        ).fetchall()
        for (wkt,) in swe_u:
            g = translate(load_wkt(wkt), xoff=offset_deg, yoff=offset_deg)
            folium.GeoJson(g.__geo_interface__,
                style_function=lambda x: {"color": "#f472b6", "weight": 1.5, "opacity": 0.7}).add_to(fg_swe_u)
    except Exception as _e:
        print("undirected Sweden layer skipped:", _e)
    fg_osm_u.add_to(m)
    fg_swe_u.add_to(m)

    # --- Invalid-path groups: 1:M / M:1 whose members don't chain into a single path ---
    from collections import Counter as _Ctr
    def _single_path(geoms, snap=1.0):
        if len(geoms) < 2:
            return True
        deg = _Ctr(); par = {}
        def find(x):
            par.setdefault(x, x)
            while par[x] != x:
                par[x] = par[par[x]]; x = par[x]
            return x
        for w in geoms:
            c = list(load_wkt(w).coords)
            p0 = (round(c[0][0] / snap) * snap, round(c[0][1] / snap) * snap)
            p1 = (round(c[-1][0] / snap) * snap, round(c[-1][1] / snap) * snap)
            deg[p0] += 1; deg[p1] += 1; par[find(p0)] = find(p1)
        ns = list(deg)
        if len({find(n) for n in ns}) != 1:
            return False
        if any(deg[n] not in (1, 2) for n in ns):
            return False
        return sum(1 for n in ns if deg[n] == 1) == 2
    _pa = {r[0]: r[1] for r in matcher.conn.execute(
        f"SELECT {matcher.columns_a['id']}, ST_AsText(ST_Transform({matcher.columns_a['geom']},'EPSG:4326','EPSG:3006')) FROM {matcher.source_a}").fetchall()}
    _pb = {r[0]: r[1] for r in matcher.conn.execute(
        f"SELECT {matcher.columns_b['id']}, ST_AsText(ST_Transform({matcher.columns_b['geom']},'EPSG:4326','EPSG:3006')) FROM {matcher.source_b}").fetchall()}
    _i1 = [(a, gg["b_id"].tolist()) for a, gg in sym.groupby("a_id")
           if len(gg) > 1 and not _single_path([_pb[b] for b in gg["b_id"] if b in _pb])]
    _i2 = [(bb, gg["a_id"].tolist()) for bb, gg in sym.groupby("b_id")
           if len(gg) > 1 and not _single_path([_pa[a] for a in gg["a_id"] if a in _pa])]
    fg_i1 = folium.FeatureGroup(name=f"❌ Invalid 1:M splits ({len(_i1)})", show=False)
    for _a, _bs in _i1:
        _t = f"1:M INVALID | A {_a} -> B {_bs}"
        if _a in gdf_a.index:
            folium.GeoJson(gdf_a.loc[_a, "geometry"].__geo_interface__,
                style_function=lambda x: {"color": "#2563eb", "weight": 5, "opacity": 0.9}, tooltip=_t).add_to(fg_i1)
        for _b in _bs:
            if _b in gdf_b.index:
                folium.GeoJson(gdf_b.loc[_b, "geom_shift"].__geo_interface__,
                    style_function=lambda x: {"color": "#ef4444", "weight": 3, "opacity": 0.9}, tooltip=_t).add_to(fg_i1)
    fg_i2 = folium.FeatureGroup(name=f"❌ Invalid M:1 merges ({len(_i2)})", show=False)
    for _bb, _as in _i2:
        _t = f"M:1 INVALID | B {_bb} <- A {_as}"
        if _bb in gdf_b.index:
            folium.GeoJson(gdf_b.loc[_bb, "geom_shift"].__geo_interface__,
                style_function=lambda x: {"color": "#16a34a", "weight": 5, "opacity": 0.9}, tooltip=_t).add_to(fg_i2)
        for _a in _as:
            if _a in gdf_a.index:
                folium.GeoJson(gdf_a.loc[_a, "geometry"].__geo_interface__,
                    style_function=lambda x: {"color": "#f59e0b", "weight": 3, "opacity": 0.9}, tooltip=_t).add_to(fg_i2)
    fg_i1.add_to(m)
    fg_i2.add_to(m)

    # Area boundary outline
    import os as _os
    if boundary_path and _os.path.exists(boundary_path):
        folium.GeoJson(
            boundary_path, name="Area boundary",
            style_function=lambda x: {"color": "#fbbf24", "weight": 2.5, "fill": False, "dashArray": "6, 4"},
        ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds([[bnds[1], bnds[0]], [bnds[3], bnds[2]]])
    return m


if not sym.empty:
    sym_map = visualize_symmetric(sm, sym)
    sym_out = "../output/conflation_symmetric_map.html"
    sym_map.save(sym_out)
    print(f"✅ Saved symmetric map to: {sym_out}")
    display(sym_map)
else:
    print("No symmetric matches to visualize.")